# 01 · Retrieval medido: la escalera de recall@5

Este notebook ejecuta y mide los cuatro escalones de mejora del retrieval sobre las preguntas con ancla de texto de **dos golden sets**: el **propio** (`golden/golden_set.jsonl`, el de la tabla del informe) y el **oficial** (`golden/oficial_20.jsonl`, para comparar con el material del curso). Cada uno aporta 13 anclas; la decisión final se toma con las 26. Reglas del juego:

- **Medición a nivel de componente**, no del agente entero: conjunto fijo de consultas → búsqueda → ¿el ancla aparece en el top-k? Aísla el retrieval del ruido del modelo y no cuesta ni un céntimo (salvo el escalón D, que reescribe consultas).
- **Un cambio, una medida.** Cada escalón añade exactamente un arreglo sobre el anterior; si la métrica se mueve, sabemos por qué.
- **recall@5** como métrica (K=5 fijado por el proyecto), con **posición del ancla** como diagnóstico de los fallos.
- El ancla es una **frase literal** del informe, no un `chunk_id`: la métrica sobrevive a cualquier re-troceado futuro.

Importante: este notebook **no toca la herramienta del agente**. `search_filings` sigue usando la búsqueda densa con filtros hasta que el baseline esté congelado; la configuración ganadora de aquí se conectará después, como mejora medida.

In [1]:
import json
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

from agente import datos, evaluadores, metricas, retrieval

# Los dos golden con ancla de texto. Los ids no chocan (pr-* frente a of-*).
GOLDENS = {
    "propio": evaluadores.cargar_golden(RAIZ / "golden/golden_set.jsonl"),
    "oficial": evaluadores.cargar_golden(RAIZ / "golden/oficial_20.jsonl"),
}
ANCLADAS = {nombre: [g for g in items if g.get("ancla_texto")]
            for nombre, items in GOLDENS.items()}
for nombre, items in ANCLADAS.items():
    print(f"Golden {nombre}: {len(GOLDENS[nombre])} preguntas · con ancla: {len(items)} "
          f"({sum(1 for g in items if g['familia']=='extractiva')} extractivas, "
          f"{sum(1 for g in items if g['familia']=='comparativa')} comparativas)")


def medir(etiqueta, buscar):
    """Recall@5 de una configuración de búsqueda sobre cada golden.
    Devuelve ({golden: recall}, {golden: {id: acierto}})."""
    recall, detalle = {}, {}
    for nombre, items in ANCLADAS.items():
        recall[nombre], detalle[nombre] = metricas.recall_en_k(items, buscar)
        ok, n = sum(detalle[nombre].values()), len(detalle[nombre])
        print(f"{etiqueta:<28}[{nombre:<7}] recall@5 = {recall[nombre]:.2f}   ({ok}/{n})")
        print("    fallos:", sorted(i for i, v in detalle[nombre].items() if not v))
    return recall, detalle


def total(detalle):
    """Recall sobre todas las anclas de todos los golden: (recall, n)."""
    ok = sum(sum(d.values()) for d in detalle.values())
    n = sum(len(d) for d in detalle.values())
    return ok / n, n


datos.cargar_indice()   # calienta índice + codificador (descarga BGE la 1ª vez)
print("Índice y codificador cargados.")

Golden propio: 20 preguntas · con ancla: 13 (7 extractivas, 6 comparativas)
Golden oficial: 20 preguntas · con ancla: 13 (6 extractivas, 7 comparativas)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Índice y codificador cargados.


## Escalón A — denso plano

El punto de partida del día 10: la pregunta **en español, tal cual**, contra todo el índice, sin filtros. Es deliberadamente ingenuo — es lo que hay antes de arreglar nada, y su recall es la línea desde la que se mide todo lo demás.

La expectativa es un recall bajo, y conviene entender por qué antes de verlo: el codificador (`bge-small-en-v1.5`) es **monolingüe en inglés** y el corpus también; una consulta en español cae lejos en el espacio de embeddings aunque pregunte exactamente por lo que el ancla dice. No es código roto — es un desajuste de idioma, y esa distinción decide qué arreglo funciona (el D) y cuáles no pueden funcionar solos (el B y el C).

In [2]:
def escalon_A(item):
    """Pregunta cruda, sin filtros: el retrieval del dia 10."""
    return retrieval.buscar_densa(item["pregunta"], k=metricas.K)

recall_A, detalle_A = medir("A · denso plano", escalon_A)

A · denso plano             [propio ] recall@5 = 0.23   (3/13)
    fallos: ['pr-c09', 'pr-c10', 'pr-c11', 'pr-c12', 'pr-c13', 'pr-e15', 'pr-e16', 'pr-e18', 'pr-e19', 'pr-e20']
A · denso plano             [oficial] recall@5 = 0.31   (4/13)
    fallos: ['of-001', 'of-004', 'of-005', 'of-006', 'of-015', 'of-016', 'of-018', 'of-019', 'of-020']


## Escalón B — filtros de metadatos

Mismo buscador, pero restringido al `(ticker, fiscal_year, item)` que la pregunta declara — aquí los aporta el golden; a nivel de agente los aporta el propio modelo como argumentos de la tool, así que la simulación es fiel.

El argumento: **buscar donde hay que buscar no es lo mismo que ordenar bien**, pero elimina de golpe las colisiones entre compañías (el riesgo de IA de Microsoft y el de Meta se parecen mucho más entre sí que cualquiera de los dos a una consulta en español). El filtro no mejora el ranking dentro de la sección correcta; solo garantiza que los 5 puestos del top se gastan en ella.

In [3]:
def escalon_B(item):
    """Pregunta cruda + filtros de metadatos del propio golden."""
    return retrieval.buscar_densa(item["pregunta"], ticker=item["ticker"],
                                  fiscal_year=item["fiscal_year"],
                                  item=item.get("item_esperado"),
                                  k=metricas.K)

recall_B, detalle_B = medir("B · + filtros", escalon_B)

B · + filtros               [propio ] recall@5 = 0.38   (5/13)
    fallos: ['pr-c09', 'pr-c10', 'pr-c11', 'pr-c12', 'pr-c13', 'pr-e15', 'pr-e16', 'pr-e20']
B · + filtros               [oficial] recall@5 = 0.46   (6/13)
    fallos: ['of-001', 'of-004', 'of-005', 'of-015', 'of-016', 'of-018', 'of-019']


## Escalón C — híbrido denso + BM25 (fusión RRF)

Se añade un ranking léxico (BM25) y se fusiona con el denso mediante **Reciprocal Rank Fusion**: `RRF(d) = 1/(60+pos_densa) + 1/(60+pos_bm25)`. Fusión por posiciones y no por suma de puntuaciones, porque una similitud coseno en [-1, 1] y un score BM25 sin escala fija **no son sumables** — las posiciones sí son comparables entre listas. RRF además no tiene pesos que ajustar, y con 13 anclas por golden, ajustar pesos sería sobreajuste garantizado.

Expectativa honesta, con el precedente de clase delante: sobre las 13 consultas **en español** del golden oficial, el híbrido no movió el recall (0.54 → 0.54). Tiene lógica — si la consulta está en otro idioma, el solape léxico con el informe es tan pobre como el semántico. BM25 aporta cuando hay términos exactos que clavar (tickers, nombres de producto); su momento llega **después** de la reescritura, no antes. Si aquí tampoco se mueve, no es un fallo del arreglo: es un resultado, y va al informe como tal.

In [4]:
def escalon_C(item):
    """Hibrido RRF + filtros, consulta aun en espanol."""
    return retrieval.buscar_hibrida(item["pregunta"], ticker=item["ticker"],
                                    fiscal_year=item["fiscal_year"],
                                    item=item.get("item_esperado"),
                                    k=metricas.K)

recall_C, detalle_C = medir("C · + híbrido RRF", escalon_C)

C · + híbrido RRF           [propio ] recall@5 = 0.46   (6/13)
    fallos: ['pr-c09', 'pr-c10', 'pr-c11', 'pr-c12', 'pr-c13', 'pr-e15', 'pr-e16']
C · + híbrido RRF           [oficial] recall@5 = 0.46   (6/13)
    fallos: ['of-001', 'of-002', 'of-004', 'of-015', 'of-016', 'of-018', 'of-019']


## Escalón D — reescritura de consulta ES→EN

El arreglo que ataca el cuello real: cada pregunta se reescribe **una vez** como consulta de búsqueda en inglés, con vocabulario de 10-K, usando un modelo lite a `temperature=0` (coste: una llamada mínima por consulta distinta, cacheada en proceso). Sobre la consulta reescrita se aplica todo lo anterior (filtros + híbrido): la escalera es acumulativa.

Dos aclaraciones metodológicas que conviene dejar escritas:

1. **La reescritura es pieza del arnés de medición, no de la herramienta.** El agente ya escribe sus consultas en inglés porque el system prompt se lo exige (verificado en el humo de la entrega 2: consultó `"artificial intelligence misuse third parties…"`). Meter la reescritura dentro de `search_filings` duplicaría la traducción y añadiría una llamada de modelo a cada búsqueda del agente.
2. Aquí el BM25 del escalón C **empieza a tener sentido**: con la consulta en inglés, los términos exactos del informe por fin pueden solapar.

In [5]:
import os

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

preguntas = {it["id"]: it["pregunta"] for items in ANCLADAS.values() for it in items}
reescritas = {i: retrieval.reescribir(q) for i, q in preguntas.items()}

# `reescribir` devuelve la pregunta tal cual si la API falla, y lo cachea:
# un escalón D medido así no es el escalón D. Mejor parar que guardar ruido.
sin_reescribir = [i for i, q in reescritas.items() if q == preguntas[i]]
if sin_reescribir:
    raise RuntimeError(
        f"{len(sin_reescribir)} consultas sin reescribir (¿fallo de API o de "
        f"límite de peticiones?): {sin_reescribir}. Ejecuta "
        f"retrieval.reescribir.cache_clear() y repite la celda.")

for nombre, items in ANCLADAS.items():
    for it in items[:2]:
        print(f"{it['id']}: {reescritas[it['id']]}")
print("…")

def escalon_D(item):
    """Hibrido RRF + filtros sobre la consulta reescrita en ingles."""
    return retrieval.buscar_hibrida(reescritas[item["id"]],
                                    ticker=item["ticker"],
                                    fiscal_year=item["fiscal_year"],
                                    item=item.get("item_esperado"),
                                    k=metricas.K)

print()
recall_D, detalle_D = medir("D · + reescritura ES→EN", escalon_D)

pr-e13: Alphabet maximum exposure to loss variable interest entities VIE leases
pr-e14: Meta variable interest entity consolidation accounting treatment Louisiana data center joint venture off balance sheet
of-001: NVIDIA competition China market export controls regulatory restrictions US government semiconductor chips
of-002: Microsoft generative AI internal systems security risks FY2025
…

D · + reescritura ES→EN     [propio ] recall@5 = 0.92   (12/13)
    fallos: ['pr-c13']
D · + reescritura ES→EN     [oficial] recall@5 = 0.69   (9/13)
    fallos: ['of-001', 'of-015', 'of-018', 'of-020']


## La escalera completa

In [6]:
ESCALONES = [
    ("A", "denso plano (pregunta ES, sin filtros)", recall_A, detalle_A),
    ("B", "+ filtros de metadatos", recall_B, detalle_B),
    ("C", "+ híbrido RRF", recall_C, detalle_C),
    ("D", "+ reescritura ES→EN", recall_D, detalle_D),
]

filas = []
for letra, config, recall, detalle in ESCALONES:
    for nombre in ANCLADAS:
        filas.append((nombre, letra, config, recall[nombre], len(detalle[nombre])))
    conjunto, n = total(detalle)
    filas.append(("combinado", letra, config, conjunto, n))
tabla = pd.DataFrame(filas, columns=["golden", "escalón", "configuración",
                                     "recall@5", "n_ancladas"])

display(tabla.pivot(index=["escalón", "configuración"], columns="golden",
                    values="recall@5")[["propio", "oficial", "combinado"]].round(2))

(RAIZ / "resultados").mkdir(exist_ok=True)
for nombre in ANCLADAS:
    detalle = pd.DataFrame({"A": detalle_A[nombre], "B": detalle_B[nombre],
                            "C": detalle_C[nombre], "D": detalle_D[nombre]}).sort_index()
    print(f"\nDetalle por pregunta — {nombre}:")
    display(detalle)
    (tabla[tabla.golden == nombre].drop(columns="golden")
          .to_csv(RAIZ / f"resultados/recall_escalera_{nombre}.csv", index=False))
    detalle.to_csv(RAIZ / f"resultados/recall_detalle_{nombre}.csv")
print("\nGuardado: resultados/recall_escalera_{propio,oficial}.csv · "
      "resultados/recall_detalle_{propio,oficial}.csv")

,golden,propio,oficial,combinado
escalón,configuración,,,
A,"denso plano (pregunta ES, sin filtros)",0.23,0.31,0.27
B,+ filtros de metadatos,0.38,0.46,0.42
C,+ híbrido RRF,0.46,0.46,0.46
D,+ reescritura ES→EN,0.92,0.69,0.81



Detalle por pregunta — propio:


,A,B,C,D
pr-c06,True,True,True,True
pr-c09,False,False,False,True
pr-c10,False,False,False,True
pr-c11,False,False,False,True
pr-c12,False,False,False,True
pr-c13,False,False,False,False
pr-e13,True,True,True,True
pr-e14,True,True,True,True
pr-e15,False,False,False,True
pr-e16,False,False,False,True



Detalle por pregunta — oficial:


,A,B,C,D
of-001,False,False,False,False
of-002,True,True,False,True
of-003,True,True,True,True
of-004,False,False,False,True
of-005,False,False,True,True
of-006,False,True,True,True
of-014,True,True,True,True
of-015,False,False,False,False
of-016,False,False,False,True
of-017,True,True,True,True



Guardado: resultados/recall_escalera_{propio,oficial}.csv · resultados/recall_detalle_{propio,oficial}.csv


## Diagnóstico de los fallos que quedan

Para cada pregunta que el escalón D no recupera en el top-5: **¿en qué puesto del ranking completo está el ancla?** La distinción importa porque decide la siguiente mejora. Un ancla en el puesto 6-15 es un problema de *orden* — candidata a re-ranking con cross-encoder o a subir k (la puerta que la propia S2 deja abierta en la celda 14 con el caso of-001). Un ancla en el puesto 200 es un problema de *representación* — ni reordenar ni subir k la salvan, y habría que mirar el troceado o el embedding.

In [7]:
hay_fallos = False
for nombre, items in ANCLADAS.items():
    for it in items:
        if detalle_D[nombre][it["id"]]:
            continue
        hay_fallos = True
        ranking = retrieval.buscar_hibrida(reescritas[it["id"]],
                                           ticker=it["ticker"],
                                           fiscal_year=it["fiscal_year"],
                                           item=it.get("item_esperado"),
                                           k=10_000)   # ranking completo del subconjunto filtrado
        pos = metricas.posicion_del_ancla(ranking, it["ancla_texto"])
        print(f"[{nombre}] {it['id']}: ancla en el puesto {pos}"
              f"   ·   consulta: {reescritas[it['id']][:60]}…"
              f"   ·   {it['pregunta'][:60]}…")
if not hay_fallos:
    print("Sin fallos en el escalón D.")

[propio] pr-c13: ancla en el puesto 6   ·   consulta: Alphabet operating cash flow 2024 2025 technical infrastruct…   ·   ¿Cómo varió el flujo de caja operativo de Alphabet entre 202…
[oficial] of-001: ancla en el puesto 12   ·   consulta: NVIDIA competition China market export controls regulatory r…   ·   ¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia e…
[oficial] of-015: ancla en el puesto 7   ·   consulta: NVIDIA revenue increase growth fiscal year 2025 compared to …   ·   ¿Cuánto creció el revenue de NVIDIA entre FY2024 y FY2025, y…
[oficial] of-018: ancla en el puesto 6   ·   consulta: Alphabet revenues growth 2025 vs 2024 capital expenditures a…   ·   ¿Cuánto crecieron los ingresos de Alphabet entre 2024 y 2025…
[oficial] of-020: ancla en el puesto 6   ·   consulta: Meta Platforms net income drivers 2025 versus 2024 discussio…   ·   ¿Cómo varió el beneficio neto de Meta entre 2024 y 2025, y q…


In [8]:
# Comprobación B vs C sobre las consultas REESCRITAS: ¿aporta BM25 algo
# cuando la consulta ya está en inglés, que es como consulta el agente?
def densa_reescrita(item):
    return retrieval.buscar_densa(reescritas[item["id"]], ticker=item["ticker"],
                                  fiscal_year=item["fiscal_year"],
                                  item=item.get("item_esperado"), k=metricas.K)

recall_Dp, detalle_Dp = medir("densa+filtros+reescritura", densa_reescrita)
print()
print("(híbrida+filtros+reescritura = escalón D, ya medido arriba)\n")

for nombre in ANCLADAS:
    difieren = sorted(i for i in detalle_D[nombre] if detalle_D[nombre][i] != detalle_Dp[nombre][i])
    for i in difieren:
        print(f"  [{nombre}] {i}: densa={'OK' if detalle_Dp[nombre][i] else 'X'}   "
              f"híbrida={'OK' if detalle_D[nombre][i] else 'X'}")
    if not difieren:
        print(f"  [{nombre}] idénticas pregunta a pregunta: BM25 ni suma ni resta en inglés.")

conj_d, n_total = total(detalle_Dp)
conj_h, _ = total(detalle_D)
print(f"\nSobre las {n_total} anclas de ambos golden:  densa {conj_d:.2f}  ·  híbrida {conj_h:.2f}")

for nombre in ANCLADAS:
    (pd.DataFrame({"config": ["densa+filtros+reescritura", "hibrida+filtros+reescritura"],
                   "recall@5": [recall_Dp[nombre], recall_D[nombre]]})
       .assign(n_ancladas=len(detalle_D[nombre]))
       .to_csv(RAIZ / f"resultados/densa_vs_hibrida_reescritas_{nombre}.csv", index=False))
print("\nGuardado: resultados/densa_vs_hibrida_reescritas_{propio,oficial}.csv")

densa+filtros+reescritura   [propio ] recall@5 = 0.85   (11/13)
    fallos: ['pr-c11', 'pr-c13']
densa+filtros+reescritura   [oficial] recall@5 = 0.77   (10/13)
    fallos: ['of-001', 'of-018', 'of-020']

(híbrida+filtros+reescritura = escalón D, ya medido arriba)

  [propio] pr-c11: densa=X   híbrida=OK
  [oficial] of-015: densa=OK   híbrida=X

Sobre las 26 anclas de ambos golden:  densa 0.81  ·  híbrida 0.81

Guardado: resultados/densa_vs_hibrida_reescritas_{propio,oficial}.csv


## Prueba ampliada: modelos de reescritura y de embeddings

La escalera fija dos piezas: el LLM lite que reescribe la consulta (`gemini-3.5-flash-lite`) y el embedding local (`bge-small-en-v1.5`). Esta celda las varía a la vez: se intentan **5 LLM de reescritura × 5 modelos de embeddings**, cada par con búsqueda densa y con híbrida RRF (ambas con filtros), sobre las **26 anclas** de los dos golden. Se añade como referencia la consulta en español sin reescribir.

Los LLM que no completan las 26 reescrituras (los dos gratuitos, por límite de peticiones) quedan fuera de la medición en lugar de medirse a medias. En la última ejecución completaron tres: `deepseek-v4-flash`, `gemini-3.5-flash-lite` (el control) y `gemini-3.8-flash`, que es el **modelo del propio agente** y por tanto la "reescritura con el propio modelo" que menciona el enunciado.

Las reescrituras y los índices de cada embedding se cachean en `corpus/cache_modelos` (derivado, fuera de git) y el coste de la prueba se contabiliza aparte. Resultados en `resultados/matriz_modelos_{recall,detalle,consultas}.csv`.

In [9]:
# Prueba ampliada: 5 LLM de reescritura x 5 modelos de embeddings.
# Precios en $/M tokens del catalogo publico de OpenRouter (19/09/2026).
import hashlib
import json
import os
import re
import threading
import time

import numpy as np
import requests

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

K = metricas.K
CACHE = RAIZ / "corpus" / "cache_modelos"      # derivado, fuera de git
SALIDA = RAIZ / "resultados"
CACHE.mkdir(parents=True, exist_ok=True)
SALIDA.mkdir(exist_ok=True)

SIN_LLM = "ES sin reescribir"
LLMS = {   # etiqueta: (modelo, $/M entrada, $/M salida, codigo corto)
    "gemma-4-31b · free":          ("openrouter:google/gemma-4-31b-it:free", 0.0, 0.0, "gemma"),
    "qwen3.8-27b · free":          ("openrouter:qwen/qwen3.8-27b:free", 0.0, 0.0, "qwen"),
    "deepseek-v4-flash":           ("openrouter:deepseek/deepseek-v4-flash", 0.04, 0.08, "dsv4"),
    "gemini-3.5-flash-lite (ctl)": (retrieval.MODELO_REESCRITURA, 0.30, 2.50, "g3.5"),
    "gemini-3.8-flash":            ("openrouter:google/gemini-3.8-flash", 0.75, 3.75, "g3.8"),
}
INSTRUCCION_QWEN = ("Instruct: Given a question about a company's SEC 10-K filing, "
                    "retrieve the passages that answer it\nQuery: ")
EMBEDDINGS = {   # etiqueta: (id en OpenRouter | None si es local, $/M, prefijo de consulta, codigo)
    "bge-small (ctl, local)": (None, 0.0, datos.PREFIJO_CONSULTA_BGE, "bge-s"),
    "nemotron-embed-1b · free": ("nvidia/nemotron-3-embed-1b:free", 0.0, "", "nemo"),
    "bge-m3": ("baai/bge-m3", 0.01, "", "m3"),
    "qwen3-embedding-8b": ("qwen/qwen3-embedding-8b", 0.01, INSTRUCCION_QWEN, "qw8b"),
    "gemini-embedding-001": ("google/gemini-embedding-001", 0.15, "", "gem"),
}
MODOS = {"densa": "densa + filtros", "hibrida": "híbrida RRF + filtros"}

ITEMS = {it["id"]: (nombre, it) for nombre, its in ANCLADAS.items() for it in its}
PREGUNTAS = {i: it["pregunta"] for i, (_, it) in ITEMS.items()}
_, meta, _ = datos.cargar_indice()
TEXTOS = meta["texto"].tolist()
HUELLA = hashlib.md5("\n".join(TEXTOS).encode()).hexdigest()[:10]   # invalida la caché si cambia el corpus
_slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


# ---------------------------------------------------------------- LLM: reescritura
def _texto(contenido):
    if isinstance(contenido, list):
        contenido = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido)
    return str(contenido).strip().strip('"').strip()


def _transitorio(e):
    """¿Merece reintento? Límites de peticiones y caídas del proveedor sí; clave o saldo no."""
    t = f"{type(e).__name__} {e}".lower()
    return any(s in t for s in ("429", "toomanyrequests", "rate", "overloaded", "502", "503", "timeout",
                                "sin respuesta", "provider returned error"))


LIMITE_S = 45      # segundos máximos por llamada
PRESUPUESTO_S = 240   # segundos máximos por LLM: pasado ese tiempo se abandona y se sigue con el siguiente


def _con_limite(fn, segundos):
    """Ejecuta fn() en un hilo y se rinde a los `segundos`. El SDK de OpenRouter no respeta su
    propio timeout: sin esto, una petición atascada (típico en los :free) cuelga la celda."""
    res = {}

    def _run():
        try:
            res["ok"] = fn()
        except BaseException as e:                     # noqa: BLE001
            res["err"] = e

    hilo = threading.Thread(target=_run, daemon=True)
    hilo.start()
    hilo.join(segundos)
    if hilo.is_alive():
        raise TimeoutError(f"sin respuesta en {segundos:.0f} s")
    if "err" in res:
        raise res["err"]
    return res["ok"]


def _invocar(modelo, prompt):
    """Una llamada, con razonamiento apagado (si el modelo lo admite). Ante un error
    transitorio (límite de peticiones, caída, sin respuesta) reintenta con espera creciente
    (10, 20 s); un fallo de clave, de saldo o de parámetro no se reintenta y prueba la
    variante sin `reasoning`. `max_retries=0`: los reintentos los controlamos aquí."""
    from langchain.chat_models import init_chat_model
    for extra in ({"reasoning": {"enabled": False}}, {}):
        for intento in range(3):
            try:
                llm = init_chat_model(modelo, temperature=0, max_tokens=1024, max_retries=0, **extra)
                return _con_limite(lambda: llm.invoke(prompt), LIMITE_S)
            except Exception as e:                     # noqa: BLE001
                ultimo = e
                if not _transitorio(e):
                    break
                time.sleep(10 * (intento + 1))
        else:
            raise ultimo                               # límite agotado: otra variante no ayuda
    raise ultimo


def reescribir_todas(modelo, pausa=0.0):
    """({id: consulta}, info). Cachea en disco; `pausa` s entre llamadas; se rinde tras 2 fallos seguidos."""
    ruta = CACHE / f"reesc_{_slug(modelo)}.json"
    cache = json.loads(ruta.read_text("utf-8")) if ruta.exists() else {}
    out, info = {}, dict(tin=0, tout=0, seg=0.0, llamadas=0, error=None)
    seguidos, inicio = 0, time.time()
    for i, q in PREGUNTAS.items():
        if time.time() - inicio > PRESUPUESTO_S:
            info["error"] = f"presupuesto de {PRESUPUESTO_S} s agotado"
            break
        clave = hashlib.md5((retrieval._PROMPT_REESCRITURA + q).encode()).hexdigest()[:16]
        if clave in cache:
            out[i] = cache[clave]
            print("·", end="", flush=True)
            continue
        try:
            t0 = time.time()
            salida = _invocar(modelo, retrieval._PROMPT_REESCRITURA.format(q=q))
            consulta = _texto(salida.content)
            if not consulta or consulta == q:
                raise RuntimeError("respuesta vacía o igual a la pregunta")
            uso = getattr(salida, "usage_metadata", None) or {}
            info["tin"] += uso.get("input_tokens", 0)
            info["tout"] += uso.get("output_tokens", 0)
            info["seg"] += time.time() - t0
            info["llamadas"] += 1
            out[i] = cache[clave] = consulta
            ruta.write_text(json.dumps(cache, ensure_ascii=False), "utf-8")   # sobrevive a un Ctrl+C
            seguidos = 0
            print("·", end="", flush=True)
            time.sleep(pausa)
        except Exception as e:                         # noqa: BLE001
            seguidos += 1
            print("✗", end="", flush=True)
            info["error"] = f"{type(e).__name__}: {str(e)[:160]}"
            if seguidos >= 2:
                break
    if len(out) == len(PREGUNTAS):
        info["error"] = None
    return out, info


# ---------------------------------------------------------------- embeddings
def _normalizar(m):
    return (m / np.linalg.norm(m, axis=1, keepdims=True)).astype("float32")


def _post_embeddings(modelo, textos):
    for intento in range(3):
        r = requests.post("https://openrouter.ai/api/v1/embeddings",
                          headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
                          json={"model": modelo, "input": textos}, timeout=120)
        if r.status_code in (429, 502, 503):
            time.sleep(5 * (intento + 1))
            continue
        break
    js = r.json() if r.status_code == 200 else {}
    if "data" not in js:
        raise RuntimeError(f"{modelo}: HTTP {r.status_code} {r.text[:160]}")
    filas = sorted(js["data"], key=lambda d: d["index"])
    return (np.array([f["embedding"] for f in filas], dtype="float32"),
            js.get("usage", {}).get("total_tokens", 0))


def _por_lotes(modelo, textos, lote=32):
    partes, tokens = [], 0
    for ini in range(0, len(textos), lote):
        v, t = _post_embeddings(modelo, textos[ini:ini + lote])
        partes.append(v)
        tokens += t
    return _normalizar(np.vstack(partes)), tokens


def matriz_corpus(api):
    """(vectores del corpus, tokens facturados). El control reutiliza el índice FAISS congelado."""
    if api is None:
        indice, _, _ = datos.cargar_indice()
        return indice.reconstruct_n(0, indice.ntotal).astype("float32"), 0
    ruta = CACHE / f"docs_{_slug(api)}_{HUELLA}.npy"
    if ruta.exists():
        return np.load(ruta), 0
    D, tokens = _por_lotes(api, TEXTOS)
    np.save(ruta, D)
    return D, tokens


def codificar_consultas(api, prefijo, textos):
    if api is None:
        return datos.codificar(textos)                 # ya lleva el prefijo BGE
    return _por_lotes(api, [prefijo + t for t in textos])[0]


# ---------------------------------------------------------------- búsqueda (numpy)
_PERMITIDAS, _BM25 = {}, {}


def _permitidas(it):
    clave = (it["ticker"], it["fiscal_year"], it.get("item_esperado"))
    if clave not in _PERMITIDAS:
        _PERMITIDAS[clave] = np.array(sorted(retrieval._posiciones_permitidas(meta, *clave)), dtype=int)
    return _PERMITIDAS[clave]


def _puestos(puntuaciones):
    puestos = np.empty(len(puntuaciones))
    puestos[np.argsort(-puntuaciones, kind="stable")] = np.arange(1, len(puntuaciones) + 1)
    return puestos


def top_k(modo, it, consulta, vq, D):
    """Posiciones del top-K. Misma lógica que retrieval.buscar_densa / buscar_hibrida."""
    idx = _permitidas(it)
    if idx.size == 0:
        return []
    denso = D[idx] @ vq
    if modo == "densa":
        return idx[np.argsort(-denso, kind="stable")[:K]]
    if consulta not in _BM25:
        _BM25[consulta] = np.asarray(retrieval._bm25().get_scores(retrieval._tokenizar(consulta)))
    rrf = (1.0 / (retrieval.RRF_KK + _puestos(denso))
           + 1.0 / (retrieval.RRF_KK + _puestos(_BM25[consulta][idx])))
    return idx[np.argsort(-rrf, kind="stable")[:K]]


def verificar_contra_retrieval(D):
    """El control debe dar EXACTAMENTE lo que da retrieval.py: si no, la matriz no es fiable."""
    for _, it in list(ITEMS.values())[:4]:
        q = it["pregunta"]
        vq = datos.codificar([q])[0]
        kw = dict(ticker=it["ticker"], fiscal_year=it["fiscal_year"], item=it.get("item_esperado"), k=K)
        assert [TEXTOS[p] for p in top_k("densa", it, q, vq, D)] == \
               [f["texto"] for f in retrieval.buscar_densa(q, **kw)], "densa difiere de retrieval.py"
        assert {TEXTOS[p] for p in top_k("hibrida", it, q, vq, D)} == \
               {f["texto"] for f in retrieval.buscar_hibrida(q, **kw)}, "híbrida difiere de retrieval.py"


# ---------------------------------------------------------------- ejecución
print("1/3 · Reescrituras (5 LLM)…   (· = hecha   ✗ = fallo)")
CONSULTAS, INFO_LLM = {SIN_LLM: dict(PREGUNTAS)}, {}
for etiqueta, (modelo, p_in, p_out, _c) in LLMS.items():
    print(f"   {etiqueta:<30} ", end="", flush=True)
    out, INFO_LLM[etiqueta] = reescribir_todas(modelo, pausa=3.0 if p_in == p_out == 0 else 0.0)
    if len(out) == len(PREGUNTAS):
        CONSULTAS[etiqueta] = out
    print(f" {len(out)}/{len(PREGUNTAS)}"
          + (f"   ✗ {INFO_LLM[etiqueta]['error']}" if INFO_LLM[etiqueta]["error"] else ""))

print("2/3 · Embeddings (5 modelos)…")
DOCS, INFO_EMB = {}, {}
for etiqueta, (api, precio, _pref, _c) in EMBEDDINGS.items():
    try:
        t0 = time.time()
        DOCS[etiqueta], tokens = matriz_corpus(api)
        INFO_EMB[etiqueta] = dict(dims=DOCS[etiqueta].shape[1], tokens=tokens or int(meta["n_tokens"].sum()),
                                  seg=time.time() - t0, error=None)
        print(f"   {etiqueta:<28} {DOCS[etiqueta].shape[1]} dims")
    except Exception as e:                             # noqa: BLE001
        INFO_EMB[etiqueta] = dict(dims=None, tokens=0, seg=0.0, error=f"{type(e).__name__}: {str(e)[:160]}")
        print(f"   {etiqueta:<28} ✗ {INFO_EMB[etiqueta]['error']}")
if "bge-small (ctl, local)" in DOCS:
    verificar_contra_retrieval(DOCS["bge-small (ctl, local)"])
    print("   ✓ el control reproduce retrieval.py (densa e híbrida)")

print("3/3 · Midiendo…")
RES = {}   # (modo, llm, embedding) -> {id: acierto}
for emb, (api, _p, prefijo, _c) in EMBEDDINGS.items():
    if emb not in DOCS:
        continue
    try:
        for llm, cons in CONSULTAS.items():
            ids = list(cons)
            Q = codificar_consultas(api, prefijo, [cons[i] for i in ids])
            for modo in MODOS:
                RES[(modo, llm, emb)] = {
                    i: metricas.acierta([{"texto": TEXTOS[p]} for p in
                                         top_k(modo, ITEMS[i][1], cons[i], Q[n], DOCS[emb])],
                                        ITEMS[i][1]["ancla_texto"])
                    for n, i in enumerate(ids)}
    except Exception as e:                             # noqa: BLE001
        INFO_EMB[emb]["error"] = f"consultas: {type(e).__name__}: {str(e)[:160]}"
        print(f"   {emb:<28} ✗ {INFO_EMB[emb]['error']}")
        for clave in [c for c in RES if c[2] == emb]:
            del RES[clave]

# ---------------------------------------------------------------- matrices
N = len(PREGUNTAS)
UMBRAL = int(np.ceil(0.95 * N))
FILAS = [SIN_LLM] + list(LLMS)
print(f"\nMatrices de ACIERTOS sobre {N} anclas (recall@{K} = aciertos/{N}; el 95 % exige ≥ {UMBRAL}).")
for modo, nombre in MODOS.items():
    m = pd.DataFrame({e: {l: (sum(RES[(modo, l, e)].values()) if (modo, l, e) in RES else pd.NA)
                          for l in FILAS} for e in EMBEDDINGS}).astype("Int64")
    print(f"\n── {nombre} ──")
    try:
        display(m.style.format(na_rep="—").background_gradient(cmap="RdYlGn", vmin=N * 0.3, vmax=N))
    except Exception:                                  # noqa: BLE001  (sin jinja2/matplotlib)
        display(m)

# Detalle pregunta a pregunta (híbrida): qué combinaciones aciertan cada ancla.
CORTO_L = {SIN_LLM: "ES", **{e: v[3] for e, v in LLMS.items()}}
CORTO_E = {e: v[3] for e, v in EMBEDDINGS.items()}
COLS = [(e, l) for e in EMBEDDINGS for l in FILAS if ("hibrida", l, e) in RES]
det = pd.DataFrame({(CORTO_E[e], CORTO_L[l]): {i: "✓" if RES[("hibrida", l, e)][i] else "·" for i in PREGUNTAS}
                    for e, l in COLS})
det["nº combos"] = [sum(RES[("hibrida", l, e)][i] for e, l in COLS) for i in PREGUNTAS]
det.insert(0, "golden", [ITEMS[i][0] for i in PREGUNTAS])
print(f"\n── Detalle por ancla (híbrida) · columnas = (embedding, LLM) · {len(COLS)} combinaciones ──")
with pd.option_context("display.max_columns", None, "display.width", 300):
    display(det.sort_values("nº combos"))
nadie = [i for i in PREGUNTAS if not any(RES[("hibrida", l, e)][i] for e, l in COLS)]
print(f"Anclas que NINGUNA combinación recupera (híbrida): {nadie or 'ninguna'}")

# ---------------------------------------------------------------- coste y ranking
print("\n── Coste ──")
filas_llm = []
for etiqueta, (_m, p_in, p_out, _c) in LLMS.items():
    inf = INFO_LLM[etiqueta]
    usd = (inf["tin"] * p_in + inf["tout"] * p_out) / 1e6
    n = max(inf["llamadas"], 1)
    filas_llm.append(dict(llm=etiqueta, llamadas=inf["llamadas"], tokens_in=inf["tin"], tokens_out=inf["tout"],
                          usd_prueba=round(usd, 5), usd_por_1000_consultas=round(usd / n * 1000, 3),
                          seg_por_consulta=round(inf["seg"] / n, 2), error=inf["error"]))
display(pd.DataFrame(filas_llm).set_index("llm"))
filas_emb = [dict(embedding=e, dims=i["dims"], tokens_corpus=i["tokens"],
                  usd_corpus=round(i["tokens"] * EMBEDDINGS[e][1] / 1e6, 4), error=i["error"])
             for e, i in INFO_EMB.items()]
display(pd.DataFrame(filas_emb).astype({"dims": "Int64"}).set_index("embedding"))
print("(si una reescritura o un índice estaban en caché, sus tokens/coste salen a 0 o estimados)")

rank = sorted(((sum(d.values()), modo, l, e) for (modo, l, e), d in RES.items()), reverse=True)
print(f"\nTop 8 combinaciones (de {len(RES)}):")
for ok, modo, l, e in rank[:8]:
    print(f"  {ok}/{N}  {ok / N:.2f}  {MODOS[modo]:<24} LLM={l:<30} emb={e}")
llegan = [(m, l, e) for ok, m, l, e in rank if ok >= UMBRAL]
print(f"\nCombinaciones que alcanzan el 95 % (≥ {UMBRAL}/{N}): {len(llegan)}")
print("Aviso: elegir el máximo de %d combinaciones sobre %d anclas sobreestima el resultado; "
      "confirmar en el hold-out ciego." % (len(RES), N))
CONTROL = ("hibrida", "gemini-3.5-flash-lite (ctl)", "bge-small (ctl, local)")
if CONTROL in RES and "detalle_D" in globals():   # el control debe parecerse al escalón D de arriba
    print(f"Coherencia: control híbrida {sum(RES[CONTROL].values())}/{N} · escalón D del notebook "
          f"{sum(sum(d.values()) for d in detalle_D.values())}/{N} (pueden diferir en ±1 si el LLM no es determinista)")

# ---------------------------------------------------------------- salida a disco
pd.DataFrame([dict(modo=m, llm=l, embedding=e, aciertos=sum(d.values()), n=len(d), recall=sum(d.values()) / len(d))
              for (m, l, e), d in RES.items()]).to_csv(SALIDA / "matriz_modelos_recall.csv", index=False)
pd.DataFrame([dict(modo=m, llm=l, embedding=e, id=i, golden=ITEMS[i][0], acierto=a)
              for (m, l, e), d in RES.items() for i, a in d.items()]).to_csv(SALIDA / "matriz_modelos_detalle.csv", index=False)
pd.DataFrame([dict(llm=l, id=i, consulta=c) for l, cons in CONSULTAS.items() for i, c in cons.items()]
             ).to_csv(SALIDA / "matriz_modelos_consultas.csv", index=False)
print("\nGuardado: resultados/matriz_modelos_{recall,detalle,consultas}.csv")


1/3 · Reescrituras (5 LLM)…   (· = hecha   ✗ = fallo)
   gemma-4-31b · free             ✗✗ 0/26   ✗ TooManyRequestsResponseError: Provider returned error
   qwen3.8-27b · free             ·✗·✗ 2/26   ✗ presupuesto de 240 s agotado
   deepseek-v4-flash              ·························· 26/26
   gemini-3.5-flash-lite (ctl)    ·························· 26/26
   gemini-3.8-flash               ·························· 26/26
2/3 · Embeddings (5 modelos)…
   bge-small (ctl, local)       384 dims
   nemotron-embed-1b · free     2048 dims
   bge-m3                       1024 dims
   qwen3-embedding-8b           4096 dims
   gemini-embedding-001         3072 dims
   ✓ el control reproduce retrieval.py (densa e híbrida)
3/3 · Midiendo…

Matrices de ACIERTOS sobre 26 anclas (recall@5 = aciertos/26; el 95 % exige ≥ 25).

── densa + filtros ──


,"bge-small (ctl, local)",nemotron-embed-1b · free,bge-m3,qwen3-embedding-8b,gemini-embedding-001
ES sin reescribir,11,19,15,20,21
gemma-4-31b · free,—,—,—,—,—
qwen3.8-27b · free,—,—,—,—,—
deepseek-v4-flash,18,20,19,22,22
gemini-3.5-flash-lite (ctl),19,21,20,22,22
gemini-3.8-flash,18,22,19,24,20



── híbrida RRF + filtros ──


,"bge-small (ctl, local)",nemotron-embed-1b · free,bge-m3,qwen3-embedding-8b,gemini-embedding-001
ES sin reescribir,12,13,12,13,14
gemma-4-31b · free,—,—,—,—,—
qwen3.8-27b · free,—,—,—,—,—
deepseek-v4-flash,22,22,20,24,23
gemini-3.5-flash-lite (ctl),23,23,22,22,22
gemini-3.8-flash,22,23,19,24,23



── Detalle por ancla (híbrida) · columnas = (embedding, LLM) · 20 combinaciones ──


golden bge-s                nemo                m3                qw8b                gem                nº combos
                   ES dsv4 g3.5 g3.8   ES dsv4 g3.5 g3.8 ES dsv4 g3.5 g3.8   ES dsv4 g3.5 g3.8  ES dsv4 g3.5 g3.8          
of-001  oficial     ·    ·    ·    ·    ·    ·    ·    ·  ·    ·    ·    ·    ·    ·    ·    ·   ·    ·    ·    ·         0
of-020  oficial     ✓    ✓    ·    ·    ·    ·    ·    ·  ·    ✓    ·    ·    ·    ·    ·    ·   ·    ·    ·    ·         3
of-015  oficial     ·    ✓    ·    ·    ·    ·    ·    ·  ·    ·    ·    ·    ·    ✓    ·    ✓   ·    ✓    ·    ·         4
pr-c13   propio     ·    ✓    ✓    ·    ·    ·    ✓    ✓  ·    ·    ·    ·    ·    ✓    ·    ✓   ·    ✓    ·    ✓         8
of-018  oficial     ·    ·    ✓    ✓    ·    ✓    ✓    ✓  ·    ·    ✓    ·    ·    ✓    ✓    ✓   ·    ·    ✓    ✓        11
pr-e16   propio     ·    ·    ✓    ✓    ·    ✓    ✓    ✓  ·    ·    ✓    ·    ·    ✓    ✓    ✓   ·    ✓    ✓    ✓        12
pr-c11   propio     ·    ·    ✓    ✓    ·    ✓    ✓    ✓  ·    ·    ✓    ✓    ·    ✓    ✓    ✓   ·    ✓    ✓    ✓        13
of-004  oficial     ·    ✓    ✓    ✓    ·    ✓    ✓    ✓  ·    ✓    ✓    ·    ·    ✓    ✓    ✓   ·    ✓    ✓    ✓        14
pr-c09   propio     ·    ✓    ✓    ✓    ·    ✓    ✓    ✓  ·    ✓    ✓    ✓    ·    ✓    ✓    ✓   ·    ✓    ✓    ✓        15
pr-e15   propio     ·    ✓    ✓    ✓    ·    ✓    ✓    ✓  ·    ✓    ✓    ✓    ·    ✓    ✓    ✓   ·    ✓    ✓    ✓        15
pr-c12   propio     ·    ✓    ✓    ✓    ·    ✓    ✓    ✓  ·    ✓    ✓    ✓    ·    ✓    ✓    ✓   ·    ✓    ✓    ✓        15
of-002  oficial     ·    ✓    ✓    ✓    ·    ✓    ✓    ✓  ·    ✓    ✓    ✓    ·    ✓    ✓    ✓   ·    ✓    ✓    ✓        15
pr-c10   propio     ·    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ·    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        18
pr-e13   propio     ✓    ✓    ✓    ✓    ·    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ·    ✓    ✓    ✓   ✓    ✓    ✓    ✓        18
of-019  oficial     ·    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ·    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        18
of-016  oficial     ·    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        19
of-003  oficial     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
pr-e19   propio     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
pr-e20   propio     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
pr-e14   propio     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
pr-e18   propio     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
pr-c06   propio     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
of-014  oficial     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
of-006  oficial     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
of-017  oficial     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20
of-005  oficial     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓  ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓   ✓    ✓    ✓    ✓        20

Anclas que NINGUNA combinación recupera (híbrida): ['of-001']

── Coste ──


,llamadas,tokens_in,tokens_out,usd_prueba,usd_por_1000_consultas,seg_por_consulta,error
llm,,,,,,,
gemma-4-31b · free,0,0,0,0.00000,0.000,0.00,TooManyRequestsResponseError: Provider returne...
qwen3.8-27b · free,2,249,585,0.00000,0.000,43.81,presupuesto de 240 s agotado
deepseek-v4-flash,0,0,0,0.00000,0.000,0.00,None
gemini-3.5-flash-lite (ctl),0,0,0,0.00000,0.000,0.00,None
gemini-3.8-flash,1,79,528,0.00204,2.039,6.42,None


,dims,tokens_corpus,usd_corpus,error
embedding,,,,
"bge-small (ctl, local)",384,702665,0.0000,None
nemotron-embed-1b · free,2048,702665,0.0000,None
bge-m3,1024,702665,0.0070,None
qwen3-embedding-8b,4096,702665,0.0070,None
gemini-embedding-001,3072,702665,0.1054,None


(si una reescritura o un índice estaban en caché, sus tokens/coste salen a 0 o estimados)

Top 8 combinaciones (de 40):
  24/26  0.92  híbrida RRF + filtros    LLM=gemini-3.8-flash               emb=qwen3-embedding-8b
  24/26  0.92  híbrida RRF + filtros    LLM=deepseek-v4-flash              emb=qwen3-embedding-8b
  24/26  0.92  densa + filtros          LLM=gemini-3.8-flash               emb=qwen3-embedding-8b
  23/26  0.88  híbrida RRF + filtros    LLM=gemini-3.8-flash               emb=nemotron-embed-1b · free
  23/26  0.88  híbrida RRF + filtros    LLM=gemini-3.8-flash               emb=gemini-embedding-001
  23/26  0.88  híbrida RRF + filtros    LLM=gemini-3.5-flash-lite (ctl)    emb=nemotron-embed-1b · free
  23/26  0.88  híbrida RRF + filtros    LLM=gemini-3.5-flash-lite (ctl)    emb=bge-small (ctl, local)
  23/26  0.88  híbrida RRF + filtros    LLM=deepseek-v4-flash              emb=gemini-embedding-001

Combinaciones que alcanzan el 95 % (≥ 25/26): 0
Aviso: elegir el máximo de 

In [10]:
# Veredicto sobre la matriz: qué combinación gana y con qué salvaguardas.
# Lee los CSV que deja la celda anterior, así que no depende del estado del kernel.
recall = pd.read_csv(RAIZ / "resultados/matriz_modelos_recall.csv")
detalle = pd.read_csv(RAIZ / "resultados/matriz_modelos_detalle.csv")

SIN_LLM = "ES sin reescribir"
CONTROL = ("hibrida", "gemini-3.5-flash-lite (ctl)", "bge-small (ctl, local)")    # lo que ya usa el repo, con híbrida
BASELINE = ("densa", "gemini-3.5-flash-lite (ctl)", "bge-small (ctl, local)")     # el baseline congelado
MARGEN_ADOPCION = 2    # anclas de ventaja sobre el control para cambiar de embedding (criterio fijado al leer la matriz)
NOMBRE_MODO = {"densa": "densa + filtros", "hibrida": "híbrida RRF + filtros"}

n = int(recall.n.iloc[0])
cand = recall[recall.llm != SIN_LLM].copy()        # solo combinaciones con consulta reescrita en inglés


def aciertos(k):
    f = recall[(recall.modo == k[0]) & (recall.llm == k[1]) & (recall.embedding == k[2])]
    return int(f.aciertos.iloc[0])


def por_golden(k):
    f = detalle[(detalle.modo == k[0]) & (detalle.llm == k[1]) & (detalle.embedding == k[2])]
    return f.groupby("golden").acierto.sum().astype(int).to_dict()


def ids(k, valor):
    f = detalle[(detalle.modo == k[0]) & (detalle.llm == k[1]) & (detalle.embedding == k[2])]
    return set(f[f.acierto == valor].id)


# desempate prefijado: a igualdad de aciertos, el embedding local (sin API, sin coste)
orden = cand.assign(local=cand.embedding.str.contains("local")).sort_values(
    ["aciertos", "local"], ascending=[False, False])
g = orden.iloc[0]
GANADORA = (g.modo, g.llm, g.embedding)
mejor = int(g.aciertos)
local = "local" in g.embedding

print(f"Top 5 de {len(cand)} combinaciones con consulta reescrita (sobre {n} anclas):")
for _, r in orden.head(5).iterrows():
    print(f"  {int(r.aciertos):>2}/{n}  {r.aciertos / n:.2f}  {NOMBRE_MODO[r.modo]:<22} "
          f"emb={r.embedding:<26} LLM={r.llm}")

print("\n" + "=" * 78)
print(f"COMBINACIÓN GANADORA: {NOMBRE_MODO[g.modo]}  ·  embedding = {g.embedding}  ·  LLM = {g.llm}")
print(f"  recall@5 = {mejor / n:.2f} ({mejor}/{n})   ·   control {aciertos(CONTROL)}/{n}   "
      f"·   baseline {aciertos(BASELINE)}/{n}")
print("=" * 78)

print("\nSalvaguardas")
# 1 · maldición del ganador
print(f"  1. Elegida como el máximo de {len(cand)} combinaciones sobre {n} anclas: el mejor de tantos "
      f"resultados sobreestima el rendimiento real.\n     Se confirma en el hold-out ciego, sin re-optimizar.")

# 2 · margen frente al ruido
cerca = int((cand.aciertos >= mejor - 1).sum())
ventaja = mejor - aciertos(CONTROL)
print(f"  2. {cerca} de {len(cand)} combinaciones quedan a una ancla o menos de la mejor: empate técnico "
      f"(1 ancla = ±{1 / n:.3f}).\n     Ventaja sobre el control: {ventaja:+d} ancla(s); "
      f"{'dentro del ruido' if ventaja < MARGEN_ADOPCION else 'por encima del ruido'}.")

# 3 · robustez al LLM
mismo = cand[(cand.modo == g.modo) & (cand.embedding == g.embedding)]
ctrl_par = cand[(cand.modo == CONTROL[0]) & (cand.embedding == CONTROL[2])]
print("  3. La misma pareja (modo, embedding) con cada LLM: "
      + ", ".join(f"{r.llm.split(' (')[0]} {int(r.aciertos)}/{n}" for _, r in mismo.iterrows())
      + f"  →  media {mismo.aciertos.mean():.1f} (control: {ctrl_par.aciertos.mean():.1f}).")

# 4 · robustez del modo
p = cand.pivot_table(index=["embedding", "llm"], columns="modo", values="aciertos")
es = recall[recall.llm == SIN_LLM].pivot_table(index="embedding", columns="modo", values="aciertos")
hib_ok = bool((p.hibrida >= p.densa).all())
print(f"  4. Con consulta en inglés la híbrida iguala o supera a la densa en {int((p.hibrida >= p.densa).sum())} "
      f"de {len(p)} parejas (embedding, LLM) y la supera en {int((p.hibrida > p.densa).sum())}.\n"
      f"     Con la consulta en español empeora en {int((es.hibrida < es.densa).sum())} de {len(es)} embeddings: "
      f"BM25 necesita solape léxico.")

# 5 · consistencia entre golden e intercambios
pg_g, pg_c = por_golden(GANADORA), por_golden(CONTROL)
peor = [k for k in pg_g if pg_g[k] < pg_c.get(k, 0)]
print("  5. Por golden (ganadora / control): "
      + ", ".join(f"{k} {pg_g[k]}/{pg_c.get(k, 0)}" for k in pg_g)
      + ("  →  pierde frente al control en " + str(peor) if peor else "  →  no pierde en ninguno")
      + f".\n     Frente al control gana {sorted(ids(GANADORA, True) - ids(CONTROL, True)) or 'ninguna'} "
      f"y pierde {sorted(ids(CONTROL, True) - ids(GANADORA, True)) or 'ninguna'}; "
      f"no recupera {sorted(ids(GANADORA, False))}.")

# 6 · coste de adopción
if local:
    print("  6. El embedding es local: no añade dependencias ni coste.")
else:
    print(f"  6. {g.embedding} no es local: cada consulta exigiría una llamada a la API de embeddings y un índice "
          f"propio\n     en corpus/cache_modelos (fuera de git) que el clon limpio del día 24 tendría que regenerar. "
          f"retrieval.py hoy solo carga el índice bge-small entregado.")

print("\nDecisión para search_filings (criterio de adopción fijado al leer la matriz, no antes):")
modo_ok = "hibrida" if hib_ok else "densa"
print(f"  · Modo: {NOMBRE_MODO[modo_ok]}"
      + (" (nunca pierde frente a la densa con consulta en inglés)." if hib_ok else "."))
print("      buscar_agente = " + ("buscar_hibrida" if hib_ok else "buscar_densa"))
if local or ventaja >= MARGEN_ADOPCION:
    print(f"  · Embedding: {g.embedding} (ventaja de {ventaja:+d} ancla(s) sobre el control).")
else:
    print(f"  · Embedding: se mantiene bge-small local. La ganadora ({g.embedding}) queda como candidata a "
          f"confirmar en el hold-out:\n    su ventaja ({ventaja:+d}) no llega a {MARGEN_ADOPCION} anclas.")
print("  · La reescritura queda fuera de la tool: el agente ya consulta en inglés por system prompt.")

Top 5 de 30 combinaciones con consulta reescrita (sobre 26 anclas):
  24/26  0.92  híbrida RRF + filtros  emb=qwen3-embedding-8b         LLM=deepseek-v4-flash
  24/26  0.92  densa + filtros        emb=qwen3-embedding-8b         LLM=gemini-3.8-flash
  24/26  0.92  híbrida RRF + filtros  emb=qwen3-embedding-8b         LLM=gemini-3.8-flash
  23/26  0.88  híbrida RRF + filtros  emb=bge-small (ctl, local)     LLM=gemini-3.5-flash-lite (ctl)
  23/26  0.88  híbrida RRF + filtros  emb=nemotron-embed-1b · free   LLM=gemini-3.5-flash-lite (ctl)

COMBINACIÓN GANADORA: híbrida RRF + filtros  ·  embedding = qwen3-embedding-8b  ·  LLM = deepseek-v4-flash
  recall@5 = 0.92 (24/26)   ·   control 23/26   ·   baseline 19/26

Salvaguardas
  1. Elegida como el máximo de 30 combinaciones sobre 26 anclas: el mejor de tantos resultados sobreestima el rendimiento real.
     Se confirma en el hold-out ciego, sin re-optimizar.
  2. 8 de 30 combinaciones quedan a una ancla o menos de la mejor: empate técnico (1 

## Lectura y decisión

*(Cifras de la última ejecución. El LLM no es determinista y una misma configuración puede variar en 1 o 2 anclas: el control mide 23/26 en la matriz y 21/26 en el escalón D, y el escalón D del golden oficial salió 0.77 en una ejecución anterior y 0.69 en esta.)*

**La escalera.** Con `gemini-3.5-flash-lite` y `bge-small`, el recall@5 pasa en el golden propio de 0.23 → 0.38 → 0.46 → **0.92** y en el oficial de 0.31 → 0.46 → 0.46 → **0.69**. El salto grande está en la reescritura (escalón D): el cuello es el **idioma**, no el algoritmo de búsqueda.

**La matriz (5 embeddings × 3 LLM de reescritura × densa/híbrida, 26 anclas).** Son 40 combinaciones: 30 con la consulta reescrita en inglés y 10 con la consulta en español como referencia. Los tres LLM son `deepseek-v4-flash`, `gemini-3.5-flash-lite` (el control) y `gemini-3.8-flash`, el modelo del propio agente. Los dos LLM gratuitos quedaron fuera por no completar las 26 reescrituras.

> **Empate en cabeza a 24/26 (0.92) entre tres combinaciones, las tres con `qwen3-embedding-8b`:** híbrida con `deepseek-v4-flash`, densa con `gemini-3.8-flash` e híbrida con `gemini-3.8-flash`. La celda del veredicto muestra la primera de las tres.
> Control (híbrida, `bge-small` local, `gemini-3.5-flash-lite`): 23/26. Baseline congelado (densa, misma reescritura): 19/26.

**Lo que es robusto.**

1. **La reescritura es lo que manda.** Sin ella, la híbrida empeora en 4 de 5 embeddings (con `qwen3-embedding-8b` pasa de 20 a 13 aciertos y con `gemini-embedding-001` de 21 a 14): BM25 no encuentra solape léxico con una consulta en español.
2. **Con consulta en inglés, la híbrida no pierde nunca frente a la densa:** iguala o supera en las 15 parejas (embedding, LLM) y supera en 11.
3. **La híbrida rinde sobre todo con un embedding débil, que es el caso del local.** Con `bge-small` y el modelo del propio agente pasa de 18 a 22 aciertos. Con `qwen3-embedding-8b`, en cambio, la densa ya iguala (24 y 24). Es la forma barata de cerrar la brecha sin pagar un embedding.

**Lo que es frágil: la elección del embedding.**

- Ocho de las 30 combinaciones con reescritura quedan a una ancla o menos de la mejor (tres de 24 y cinco de 23): **empate técnico**. Una ancla vale ±0,038.
- Las tres de cabeza superan al control en **una sola ancla** (of-015, del golden oficial). En el golden propio empatan a 13/13 con él.
- La ventaja depende de quién escribe la consulta: la pareja (híbrida, `qwen3-embedding-8b`) da 24/26 con `deepseek-v4-flash` y con `gemini-3.8-flash`, pero 22/26 con `gemini-3.5-flash-lite`.
- Con el modelo del propio agente y el embedding local, la híbrida da 22/26: una ancla menos que el control (23/26, con `gemini-3.5-flash-lite`).
- Es el máximo de 30 combinaciones sobre 26 anclas, así que sobreestima su rendimiento real (maldición del ganador).
- Adoptar `qwen3-embedding-8b` exige una llamada a la API de embeddings por consulta y regenerar un índice de 4096 dimensiones en el clon limpio del día 24, con dependencia de red. `retrieval.py` solo carga hoy el índice `bge-small` entregado.

**Decisión para `search_filings`** (criterio fijado al leer la matriz, no antes: cambiar de embedding solo si la ganadora supera al control en al menos 2 anclas).

- **Modo: híbrida + filtros** (`buscar_agente = buscar_hibrida`). Es el único hallazgo que se sostiene en todos los embeddings, y compensa la debilidad del embedding local.
- **Embedding: se mantiene `bge-small` local.** Las combinaciones de cabeza, con una ventaja de una ancla, no cumplen el criterio y quedan como candidatas para confirmar en las 10 preguntas ciegas del día 24, sin re-optimizar.
- **La reescritura queda fuera de la tool:** el agente ya consulta en inglés por system prompt, y meterla dentro duplicaría la traducción y añadiría una llamada de modelo a cada búsqueda.
- **Fallos que quedan:** of-001 no la recupera ninguna combinación y of-020 tampoco las tres de cabeza. Si se quiere subir más, la palanca es un re-ranker sobre el top-20, no otro embedding.

In [11]:
print("NOTEBOOK 01 COMPLETADO")
print("Ficheros escritos: resultados/recall_escalera_*.csv · recall_detalle_*.csv · "
      "densa_vs_hibrida_reescritas_*.csv · matriz_modelos_*.csv")
print("Siguiente: notebook 02 (congelación del baseline)")

NOTEBOOK 01 COMPLETADO
Ficheros escritos: resultados/recall_escalera_*.csv · recall_detalle_*.csv · densa_vs_hibrida_reescritas_*.csv · matriz_modelos_*.csv
Siguiente: notebook 02 (congelación del baseline)
